In [2]:
import os
os.getcwd()

'c:\\Users\\hamoud\\Desktop\\reddit_jokes_project'

In [ ]:
import os
import json
import math
import torch
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import DataLoader
from peft import PeftModel

BASELINE_PATH = "./data/outputs/baseline_samples/distilgpt2_baseline_3.jsonl"
FINETUNED_PATH = "./data/outputs/finetuned_samples/distilgpt2_lora_finetuned3.jsonl"
VAL_PATH = "./data/splits/val.jsonl"

MODEL_NAME = "distilgpt2"
CKPT_DIR = "./data/checkpoints/try3/distilgpt2_lora_finetuned3"

MAX_LEN = 256
BATCH_SIZE = 8

def distinct_n(texts, n=1):
    total_ngrams = 0
    unique_ngrams = set()

    for text in texts:
        tokens = text.split()
        ngrams = zip(*[tokens[i:] for i in range(n)])
        ngrams = list(ngrams)

        total_ngrams += len(ngrams)
        unique_ngrams.update(ngrams)

    if total_ngrams == 0:
        return 0.0

    return len(unique_ngrams) / total_ngrams


def avg_length(texts):
    lengths = [len(t.split()) for t in texts]
    return sum(lengths) / max(len(lengths), 1)


def load_predictions(path):
    preds = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            preds.append(row["prediction"])
    return preds


class JsonlDataset(torch.utils.data.Dataset):
    def __init__(self, path):
        self.rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                self.rows.append(json.loads(line))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


def collate_fn(batch, tok, max_len):
    input_ids_list = []
    attn_list = []

    for ex in batch:
        inst = ex["instruction"].strip()
        resp = ex["response"].strip()
        full = inst + "\nJoke:\n" + resp

        enc = tok(full, truncation=True, max_length=max_len, return_tensors="pt")
        input_ids_list.append(enc["input_ids"][0])
        attn_list.append(enc["attention_mask"][0])

    input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids_list, batch_first=True, padding_value=tok.pad_token_id
    )
    attn_mask = torch.nn.utils.rnn.pad_sequence(
        attn_list, batch_first=True, padding_value=0
    )

    return {"input_ids": input_ids, "attention_mask": attn_mask}


@torch.no_grad()
def compute_perplexity(model, loader, device):
    model.eval()
    total_loss = 0.0
    n = 0

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch, labels=batch["input_ids"])
        total_loss += outputs.loss.item()
        n += 1

    avg_loss = total_loss / max(n, 1)
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity

def main():
    print("Loading predictions...")

    baseline_preds = load_predictions(BASELINE_PATH)
    finetuned_preds = load_predictions(FINETUNED_PATH)

    print("\n===== Diversity Metrics =====\n")

    for name, preds in [
        ("Baseline", baseline_preds),
        ("Finetuned", finetuned_preds)
    ]:
        d1 = distinct_n(preds, 1)
        d2 = distinct_n(preds, 2)
        avg_len = avg_length(preds)

        print(f"{name}")
        print(f"  Avg length     : {avg_len:.2f} words")
        print(f"  Distinct-1     : {d1:.4f}")
        print(f"  Distinct-2     : {d2:.4f}")
        print()

    print("===== Perplexity (Validation Set) =====\n")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    tok.pad_token = tok.eos_token

    val_ds = JsonlDataset(VAL_PATH)
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, tok, MAX_LEN),
    )

    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
    base_loss, base_ppl = compute_perplexity(base_model, val_loader, device)

    base_for_lora = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
    lora_model = PeftModel.from_pretrained(base_for_lora, CKPT_DIR).to(device)

    tuned_loss, tuned_ppl = compute_perplexity(lora_model, val_loader, device)

    print("Baseline")
    print(f"  Val loss       : {base_loss:.4f}")
    print(f"  Perplexity     : {base_ppl:.2f}\n")

    print("Finetuned")
    print(f"  Val loss       : {tuned_loss:.4f}")
    print(f"  Perplexity     : {tuned_ppl:.2f}\n")

    print("===== Conclusion =====")
    print("Lower perplexity is better.")
    print("Higher Distinct-n is more diverse.")
    print("Compare both to analyze trade-offs.")


if __name__ == "__main__":
    main()

Loading predictions...

===== Diversity Metrics =====

Baseline
  Avg length     : 39.13 words
  Distinct-1     : 0.3841
  Distinct-2     : 0.8964

Finetuned
  Avg length     : 43.34 words
  Distinct-1     : 0.3449
  Distinct-2     : 0.8174

===== Perplexity (Validation Set) =====



Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Baseline
  Val loss       : 6.3671
  Perplexity     : 582.40

Finetuned
  Val loss       : 7.2273
  Perplexity     : 1376.56

===== Conclusion =====
Lower perplexity is better.
Higher Distinct-n is more diverse.
Compare both to analyze trade-offs.


In [5]:
# evaluate.py
import os
import json
import math
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# =========================
# ====== PATH CONFIG ======
# =========================

BASELINE_PATH = "./data/outputs/baseline_samples/distilgpt2_baseline_3.jsonl"
FINETUNED_PATH = "./data/outputs/finetuned_samples/distilgpt2_lora_finetuned3.jsonl"
VAL_PATH = "./data/splits/val.jsonl"

MODEL_NAME = "distilgpt2"
CKPT_DIR = "./data/checkpoints/try3/distilgpt2_lora_finetuned3"

MAX_LEN = 256
BATCH_SIZE = 8


# =========================================================
# ------------------ DIVERSITY METRICS --------------------
# =========================================================

def distinct_n(texts, n=1):
    total_ngrams = 0
    unique_ngrams = set()

    for text in texts:
        tokens = text.split()
        if len(tokens) < n:
            continue

        ngrams = zip(*[tokens[i:] for i in range(n)])
        ngrams = list(ngrams)

        total_ngrams += len(ngrams)
        unique_ngrams.update(ngrams)

    if total_ngrams == 0:
        return 0.0

    return len(unique_ngrams) / total_ngrams


def avg_length(texts):
    lengths = [len(t.split()) for t in texts]
    return sum(lengths) / max(len(lengths), 1)


def load_predictions(path):
    preds = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            preds.append(row["prediction"])
    return preds


# =========================================================
# ---------------- DATASET FOR PERPLEXITY -----------------
# =========================================================

class JsonlDataset(Dataset):
    def __init__(self, path):
        self.rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                self.rows.append(json.loads(line))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


def collate_fn(batch, tok, max_len):
    input_ids_list = []
    attn_list = []
    labels_list = []

    for ex in batch:
        inst = ex["instruction"].strip()
        resp = ex["response"].strip()

        prompt = inst + "\nJoke:\n"
        full = prompt + resp

        enc_full = tok(
            full,
            truncation=True,
            max_length=max_len,
            add_special_tokens=False
        )

        enc_prompt = tok(
            prompt,
            truncation=True,
            max_length=max_len,
            add_special_tokens=False
        )

        input_ids = enc_full["input_ids"]
        attention_mask = enc_full["attention_mask"]

        labels = input_ids.copy()
        prompt_len = len(enc_prompt["input_ids"])

        # MASK PROMPT TOKENS (critical fix)
        labels[:prompt_len] = [-100] * prompt_len

        input_ids_list.append(torch.tensor(input_ids))
        attn_list.append(torch.tensor(attention_mask))
        labels_list.append(torch.tensor(labels))

    input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids_list, batch_first=True, padding_value=tok.pad_token_id
    )
    attn_mask = torch.nn.utils.rnn.pad_sequence(
        attn_list, batch_first=True, padding_value=0
    )
    labels = torch.nn.utils.rnn.pad_sequence(
        labels_list, batch_first=True, padding_value=-100
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attn_mask,
        "labels": labels
    }


@torch.no_grad()
def compute_perplexity(model, loader, device):
    model.eval()
    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        total_loss += outputs.loss.item()
        n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity


# =========================================================
# ------------------------ MAIN ---------------------------
# =========================================================

def main():

    print("Loading predictions...\n")

    baseline_preds = load_predictions(BASELINE_PATH)
    finetuned_preds = load_predictions(FINETUNED_PATH)

    print("===== Diversity Metrics =====\n")

    for name, preds in [
        ("Baseline", baseline_preds),
        ("Finetuned", finetuned_preds)
    ]:
        d1 = distinct_n(preds, 1)
        d2 = distinct_n(preds, 2)
        avg_len = avg_length(preds)

        print(f"{name}")
        print(f"  Avg length     : {avg_len:.2f} words")
        print(f"  Distinct-1     : {d1:.4f}")
        print(f"  Distinct-2     : {d2:.4f}")
        print()

    print("===== Perplexity (Validation Set) =====\n")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device, "\n")

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    tok.pad_token = tok.eos_token

    val_ds = JsonlDataset(VAL_PATH)
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, tok, MAX_LEN)
    )

    # -------- Baseline --------
    baseline_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
    base_loss, base_ppl = compute_perplexity(baseline_model, val_loader, device)

    # -------- Finetuned --------
    base_for_lora = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
    finetuned_model = PeftModel.from_pretrained(base_for_lora, CKPT_DIR).to(device)

    tuned_loss, tuned_ppl = compute_perplexity(finetuned_model, val_loader, device)

    print("Baseline")
    print(f"  Val loss       : {base_loss:.4f}")
    print(f"  Perplexity     : {base_ppl:.2f}\n")

    print("Finetuned")
    print(f"  Val loss       : {tuned_loss:.4f}")
    print(f"  Perplexity     : {tuned_ppl:.2f}\n")

    print("===== Interpretation =====")
    print("- Lower perplexity is better.")
    print("- Higher Distinct-n means more lexical diversity.")
    print("- Small diversity drop + strong perplexity drop = successful specialization.")


if __name__ == "__main__":
    main()

Loading predictions...

===== Diversity Metrics =====

Baseline
  Avg length     : 39.13 words
  Distinct-1     : 0.3841
  Distinct-2     : 0.8964

Finetuned
  Avg length     : 43.34 words
  Distinct-1     : 0.3449
  Distinct-2     : 0.8174

===== Perplexity (Validation Set) =====

Device: cuda 



Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Baseline
  Val loss       : 4.3534
  Perplexity     : 77.74

Finetuned
  Val loss       : 3.4890
  Perplexity     : 32.75

===== Interpretation =====
- Lower perplexity is better.
- Higher Distinct-n means more lexical diversity.
- Small diversity drop + strong perplexity drop = successful specialization.
